# Grad-CAM Temporal Evaluation Statistical Testing

This notebook analyzes existing Grad-CAM temporal evaluation CSV outputs only. It does **not** rerun model inference, regenerate Grad-CAM heatmaps, retrain models, or require a GPU.

The comparisons are paired because the same EchoNet-Dynamic test samples were evaluated under multiple model/layer/stride conditions. Paired tests therefore compare within-sample differences rather than treating model outputs as independent observations.

## Statistical Plan

The Wilcoxon signed-rank test evaluates whether the median paired difference between two matched conditions is zero. It is used here because many saliency metrics are bounded, skewed, or non-Gaussian.

Multiple-comparison correction is required because each research question involves many metrics and repeated tests. Holm-Bonferroni correction controls the family-wise error rate while being less conservative than plain Bonferroni.

Matched-pairs rank-biserial correlation is reported as the paired effect size. In this notebook, the paired difference is always computed as **group B minus group A**. A positive effect size therefore means group B tends to have higher values than group A. Metric direction is interpreted separately: higher is better for saliency consistency, saliency mask overlap, and temporal saliency IoU; lower is better for saliency centroid motion.

Statistical significance is not the same as practical importance. Use p-values, effect sizes, median paired differences, and the raw descriptive statistics together.

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path("/Users/jiyoonoh/Documents/Echonet_temporal_XAI")
os.chdir(PROJECT_ROOT)

print("Working directory:", Path.cwd())

Working directory: /Users/jiyoonoh/Documents/Echonet_temporal_XAI


In [2]:
from __future__ import annotations

from pathlib import Path
import itertools
import json
import math
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy import stats

try:
    from statsmodels.stats.multitest import multipletests
    HAS_STATSMODELS = True
except Exception as exc:
    multipletests = None
    HAS_STATSMODELS = False
    warnings.warn(f"statsmodels is unavailable; falling back to local Holm correction: {exc}")

RANDOM_SEED = 42
BOOTSTRAP_ITERATIONS = 5000
ALPHA = 0.05
rng = np.random.default_rng(RANDOM_SEED)

INPUT_RUN_DIR = Path("outputs/iMIMIC_additional_experiments/iMIMIC_updated_gradcam_temporal_evaluation_07_26")
METRICS_DIR = INPUT_RUN_DIR / "metrics"
TABLES_DIR = INPUT_RUN_DIR / "tables"
OUTPUT_DIR = Path("outputs/iMIMIC_additional_experiments/iMIMIC_updated_gradcam_temporal_evaluation_statistical_analysis_07_28")
OUTPUT_TABLE_DIR = OUTPUT_DIR / "tables"
OUTPUT_FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_REPORT_DIR = OUTPUT_DIR / "reports"
for directory in [OUTPUT_DIR, OUTPUT_TABLE_DIR, OUTPUT_FIGURE_DIR, OUTPUT_REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PER_SAMPLE_PATH = METRICS_DIR / "per_sample_metrics.csv"
assert PER_SAMPLE_PATH.exists(), f"Missing required file: {PER_SAMPLE_PATH}"

per_sample = pd.read_csv(PER_SAMPLE_PATH)
comparison_files = sorted(METRICS_DIR.glob("*comparison*.csv")) + sorted(TABLES_DIR.glob("comparison*.csv"))
aggregated_paths = [p for p in [METRICS_DIR / "aggregated_metrics.csv", TABLES_DIR / "aggregated_metrics.csv"] if p.exists()]
print("Loaded per-sample metrics:", PER_SAMPLE_PATH, per_sample.shape)
print("Aggregated files:", aggregated_paths)
print("Comparison files:", comparison_files)

Loaded per-sample metrics: outputs/iMIMIC_additional_experiments/iMIMIC_updated_gradcam_temporal_evaluation_07_26/metrics/per_sample_metrics.csv (63800, 39)
Aggregated files: [PosixPath('outputs/iMIMIC_additional_experiments/iMIMIC_updated_gradcam_temporal_evaluation_07_26/metrics/aggregated_metrics.csv'), PosixPath('outputs/iMIMIC_additional_experiments/iMIMIC_updated_gradcam_temporal_evaluation_07_26/tables/aggregated_metrics.csv')]
Comparison files: [PosixPath('outputs/iMIMIC_additional_experiments/iMIMIC_updated_gradcam_temporal_evaluation_07_26/tables/comparison_across_strides.csv'), PosixPath('outputs/iMIMIC_additional_experiments/iMIMIC_updated_gradcam_temporal_evaluation_07_26/tables/comparison_across_target_layers.csv'), PosixPath('outputs/iMIMIC_additional_experiments/iMIMIC_updated_gradcam_temporal_evaluation_07_26/tables/comparison_convlstm_vs_unet.csv')]


In [3]:
# Schema inspection and automatic column identification.
print("Columns:")
for col in per_sample.columns:
    print(" -", col)

required_identifier_candidates = {
    "sample_id": ["sample_id", "id", "case_id"],
    "model_family": ["model_family", "family"],
    "model_id": ["model_id", "model", "checkpoint_id"],
    "temporal_stride": ["temporal_stride", "stride"],
    "target_layer": ["target_layer", "layer", "cam_layer"],
}

def detect_column(df: pd.DataFrame, candidates: list[str], required: bool = True) -> str | None:
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    if required:
        raise ValueError(f"Could not find any of these required columns: {candidates}")
    return None

COL_SAMPLE = detect_column(per_sample, required_identifier_candidates["sample_id"])
COL_FAMILY = detect_column(per_sample, required_identifier_candidates["model_family"])
COL_MODEL = detect_column(per_sample, required_identifier_candidates["model_id"])
COL_STRIDE = detect_column(per_sample, required_identifier_candidates["temporal_stride"])
COL_LAYER = detect_column(per_sample, required_identifier_candidates["target_layer"])

metric_candidates = {
    "saliency_consistency": ["saliency_consistency_mean"],
    "saliency_mask_overlap": ["center_saliency_mask_overlap", "dense_saliency_mask_overlap_mean"],
    "temporal_saliency_iou": ["temporal_saliency_iou_mean"],
    "saliency_centroid_motion": ["saliency_centroid_motion_mean"],
}
metric_columns = {}
for metric_name, candidates in metric_candidates.items():
    metric_columns[metric_name] = detect_column(per_sample, candidates)

metric_direction = {
    "saliency_consistency": "higher_better",
    "saliency_mask_overlap": "higher_better",
    "temporal_saliency_iou": "higher_better",
    "saliency_centroid_motion": "lower_better",
}

print("\nDetected identifier columns:")
print({
    "sample_id": COL_SAMPLE,
    "model_family": COL_FAMILY,
    "model_id": COL_MODEL,
    "temporal_stride": COL_STRIDE,
    "target_layer": COL_LAYER,
})
print("\nDetected metric columns:")
print(metric_columns)

print("\nShape:", per_sample.shape)
print("Unique model families:", sorted(per_sample[COL_FAMILY].dropna().astype(str).unique()))
print("Unique model IDs:", sorted(per_sample[COL_MODEL].dropna().astype(str).unique()))
print("Unique target layers:", sorted(per_sample[COL_LAYER].dropna().astype(str).unique()))
print("Unique strides:", sorted(per_sample[COL_STRIDE].dropna().unique()))

Columns:
 - sample_id
 - video_id
 - center_frame_idx
 - frame_indices
 - fps
 - window_span_seconds
 - model_family
 - model_id
 - temporal_stride
 - target_layer
 - raw_target_layer
 - checkpoint_path
 - n_layer_calls_captured
 - gradcam_protocol
 - mask_sources
 - center_dice
 - center_pred_iou
 - saliency_consistency_mean
 - saliency_centroid_motion_mean
 - center_saliency_mask_overlap
 - dense_saliency_mask_overlap_mean
 - dense_saliency_mask_overlap_std
 - temporal_saliency_iou_mean
 - saliency_percentile
 - saliency_mask_overlap_frame_0
 - saliency_mask_overlap_frame_1
 - saliency_mask_overlap_frame_2
 - saliency_mask_overlap_frame_3
 - saliency_mask_overlap_frame_4
 - saliency_centroid_x_frame_0
 - saliency_centroid_y_frame_0
 - saliency_centroid_x_frame_1
 - saliency_centroid_y_frame_1
 - saliency_centroid_x_frame_2
 - saliency_centroid_y_frame_2
 - saliency_centroid_x_frame_3
 - saliency_centroid_y_frame_3
 - saliency_centroid_x_frame_4
 - saliency_centroid_y_frame_4

Detecte

In [4]:
# Data validation report.
validation_lines = []
validation_lines.append(f"Input per-sample CSV: {PER_SAMPLE_PATH}")
validation_lines.append(f"Shape: {per_sample.shape}")
validation_lines.append(f"Columns: {list(per_sample.columns)}")
validation_lines.append(f"Identifier columns: sample={COL_SAMPLE}, family={COL_FAMILY}, model={COL_MODEL}, stride={COL_STRIDE}, layer={COL_LAYER}")
validation_lines.append(f"Metric columns: {metric_columns}")

id_cols = [COL_SAMPLE, COL_FAMILY, COL_MODEL, COL_STRIDE, COL_LAYER]
duplicate_mask = per_sample.duplicated(subset=id_cols, keep=False)
duplicate_rows = per_sample.loc[duplicate_mask, id_cols]
validation_lines.append(f"Duplicate rows by {id_cols}: {int(duplicate_mask.sum())}")
if duplicate_mask.any():
    validation_lines.append("Duplicate row examples:")
    validation_lines.append(duplicate_rows.head(20).to_string(index=False))

missing_summary = per_sample[id_cols + list(metric_columns.values())].isna().sum().sort_values(ascending=False)
validation_lines.append("Missing values by important column:")
validation_lines.append(missing_summary.to_string())

nonfinite_rows = []
for metric_name, col in metric_columns.items():
    values = pd.to_numeric(per_sample[col], errors="coerce").to_numpy(dtype=float)
    nonfinite = ~np.isfinite(values)
    nonfinite_rows.append({"metric": metric_name, "column": col, "nonfinite_count": int(nonfinite.sum())})
nonfinite_df = pd.DataFrame(nonfinite_rows)
validation_lines.append("Non-finite metric values:")
validation_lines.append(nonfinite_df.to_string(index=False))

# Known condition names, detected from schema.
all_layers = set(per_sample[COL_LAYER].dropna().astype(str).unique())
all_families = set(per_sample[COL_FAMILY].dropna().astype(str).unique())
required_layers = {
    "unet_final": "unet_final_convolution",
    "convlstm_decoder3": "convlstm_decoder3_sliding_window",
    "convlstm_encoder": "convlstm_bottleneck_encoder",
    "convlstm_temporal": "convlstm_temporal_bottleneck",
}
for label, layer in required_layers.items():
    validation_lines.append(f"Required layer {label}: {layer} present={layer in all_layers}")

strides = sorted(int(s) for s in per_sample[COL_STRIDE].dropna().unique())
validation_lines.append(f"Detected strides: {strides}")

# Check paired sample sets for the main planned comparisons.
def sample_set_for(layer: str, stride: int | None = None, family: str | None = None) -> set[str]:
    d = per_sample[per_sample[COL_LAYER].astype(str) == layer]
    if stride is not None:
        d = d[pd.to_numeric(d[COL_STRIDE], errors="coerce") == stride]
    if family is not None:
        d = d[d[COL_FAMILY].astype(str) == family]
    return set(d[COL_SAMPLE].astype(str))

for stride in strides:
    unet_samples = sample_set_for(required_layers["unet_final"], stride=stride, family="unet_2d")
    dec_samples = sample_set_for(required_layers["convlstm_decoder3"], stride=stride, family="convlstm_unet")
    validation_lines.append(
        f"Paired final explanations stride {stride}: unet={len(unet_samples)}, decoder3={len(dec_samples)}, matched={len(unet_samples & dec_samples)}, unet_only={len(unet_samples - dec_samples)}, decoder_only={len(dec_samples - unet_samples)}"
    )
    enc_samples = sample_set_for(required_layers["convlstm_encoder"], stride=stride, family="convlstm_unet")
    tmp_samples = sample_set_for(required_layers["convlstm_temporal"], stride=stride, family="convlstm_unet")
    validation_lines.append(
        f"Paired internal representations stride {stride}: encoder={len(enc_samples)}, temporal={len(tmp_samples)}, matched={len(enc_samples & tmp_samples)}, encoder_only={len(enc_samples - tmp_samples)}, temporal_only={len(tmp_samples - enc_samples)}"
    )

for layer in [required_layers["convlstm_decoder3"], required_layers["convlstm_encoder"], required_layers["convlstm_temporal"]]:
    sets = [sample_set_for(layer, stride=s, family="convlstm_unet") for s in strides]
    common = set.intersection(*sets) if sets else set()
    union = set.union(*sets) if sets else set()
    validation_lines.append(f"Repeated-measures stride check for {layer}: common={len(common)}, union={len(union)}, identical_all={all(x == sets[0] for x in sets) if sets else False}")

report_text = "\n".join(validation_lines)
print(report_text)
(OUTPUT_REPORT_DIR / "data_validation_report.txt").write_text(report_text, encoding="utf-8")
nonfinite_df.to_csv(OUTPUT_TABLE_DIR / "nonfinite_metric_values.csv", index=False)

Input per-sample CSV: outputs/iMIMIC_additional_experiments/iMIMIC_updated_gradcam_temporal_evaluation_07_26/metrics/per_sample_metrics.csv
Shape: (63800, 39)
Columns: ['sample_id', 'video_id', 'center_frame_idx', 'frame_indices', 'fps', 'window_span_seconds', 'model_family', 'model_id', 'temporal_stride', 'target_layer', 'raw_target_layer', 'checkpoint_path', 'n_layer_calls_captured', 'gradcam_protocol', 'mask_sources', 'center_dice', 'center_pred_iou', 'saliency_consistency_mean', 'saliency_centroid_motion_mean', 'center_saliency_mask_overlap', 'dense_saliency_mask_overlap_mean', 'dense_saliency_mask_overlap_std', 'temporal_saliency_iou_mean', 'saliency_percentile', 'saliency_mask_overlap_frame_0', 'saliency_mask_overlap_frame_1', 'saliency_mask_overlap_frame_2', 'saliency_mask_overlap_frame_3', 'saliency_mask_overlap_frame_4', 'saliency_centroid_x_frame_0', 'saliency_centroid_y_frame_0', 'saliency_centroid_x_frame_1', 'saliency_centroid_y_frame_1', 'saliency_centroid_x_frame_2', 'sa

In [5]:
# Helper functions for statistics.
def clean_numeric(series: pd.Series) -> np.ndarray:
    values = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    return values[np.isfinite(values)]


def bootstrap_mean_ci(values: np.ndarray, n_boot: int = BOOTSTRAP_ITERATIONS, seed: int = RANDOM_SEED) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    if len(values) == 1:
        return float(values[0]), float(values[0])
    local_rng = np.random.default_rng(seed)
    idx = local_rng.integers(0, len(values), size=(n_boot, len(values)))
    means = values[idx].mean(axis=1)
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def descriptive_stats(values: np.ndarray) -> dict[str, float | int]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return {
            "n": 0, "mean": np.nan, "std": np.nan, "median": np.nan,
            "q1": np.nan, "q3": np.nan, "iqr": np.nan,
            "ci95_low": np.nan, "ci95_high": np.nan,
        }
    q1, med, q3 = np.percentile(values, [25, 50, 75])
    ci_low, ci_high = bootstrap_mean_ci(values)
    return {
        "n": int(len(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values, ddof=1)) if len(values) > 1 else 0.0,
        "median": float(med),
        "q1": float(q1),
        "q3": float(q3),
        "iqr": float(q3 - q1),
        "ci95_low": ci_low,
        "ci95_high": ci_high,
    }


def holm_bonferroni(p_values: list[float], alpha: float = ALPHA) -> tuple[list[float], list[bool]]:
    p = np.asarray([np.nan if x is None else float(x) for x in p_values], dtype=float)
    adjusted = np.full_like(p, np.nan, dtype=float)
    reject = np.zeros_like(p, dtype=bool)
    valid_mask = np.isfinite(p)
    if not valid_mask.any():
        return adjusted.tolist(), reject.tolist()
    if HAS_STATSMODELS and multipletests is not None:
        reject_valid, adjusted_valid, _, _ = multipletests(p[valid_mask], alpha=alpha, method="holm")
        adjusted[valid_mask] = adjusted_valid
        reject[valid_mask] = reject_valid
        return adjusted.tolist(), reject.tolist()
    valid = np.where(valid_mask)[0]
    order = valid[np.argsort(p[valid])]
    m = len(order)
    raw_adj = np.empty(m, dtype=float)
    for rank, idx in enumerate(order):
        raw_adj[rank] = min((m - rank) * p[idx], 1.0)
    # Enforce monotonicity in sorted order.
    raw_adj = np.maximum.accumulate(raw_adj)
    for rank, idx in enumerate(order):
        adjusted[idx] = raw_adj[rank]
        reject[idx] = adjusted[idx] < alpha
    return adjusted.tolist(), reject.tolist()


def paired_rank_biserial(diff: np.ndarray) -> float:
    diff = np.asarray(diff, dtype=float)
    diff = diff[np.isfinite(diff)]
    nonzero = diff[diff != 0]
    n = len(nonzero)
    if n == 0:
        return np.nan
    ranks = stats.rankdata(np.abs(nonzero), method="average")
    w_pos = float(ranks[nonzero > 0].sum())
    w_neg = float(ranks[nonzero < 0].sum())
    denom = n * (n + 1) / 2.0
    return float((w_pos - w_neg) / denom)


def paired_wilcoxon_from_vectors(a: pd.Series, b: pd.Series, min_pairs: int = 5) -> dict[str, float | int | str | bool]:
    pair = pd.concat([pd.to_numeric(a, errors="coerce"), pd.to_numeric(b, errors="coerce")], axis=1)
    pair.columns = ["a", "b"]
    finite = np.isfinite(pair["a"].to_numpy(dtype=float)) & np.isfinite(pair["b"].to_numpy(dtype=float))
    pair = pair.loc[finite]
    diff = pair["b"].to_numpy(dtype=float) - pair["a"].to_numpy(dtype=float)
    n = len(diff)
    zero_count = int(np.sum(diff == 0))
    result = {
        "paired_n": int(n),
        "zero_differences": zero_count,
        "median_paired_difference_b_minus_a": float(np.median(diff)) if n else np.nan,
        "rank_biserial_b_minus_a": paired_rank_biserial(diff) if n else np.nan,
        "wilcoxon_statistic": np.nan,
        "p_raw": np.nan,
        "test_performed": False,
        "exclusion_reason": "",
    }
    if n < min_pairs:
        result["exclusion_reason"] = f"Too few finite matched pairs: {n} < {min_pairs}"
        return result
    if np.all(diff == 0):
        result["exclusion_reason"] = "All paired differences are zero; Wilcoxon with zero_method='wilcox' has no non-zero differences."
        result["p_raw"] = 1.0
        result["wilcoxon_statistic"] = 0.0
        return result
    try:
        w = stats.wilcoxon(pair["b"], pair["a"], alternative="two-sided", zero_method="wilcox", method="auto")
        result["wilcoxon_statistic"] = float(w.statistic)
        result["p_raw"] = float(w.pvalue)
        result["test_performed"] = True
    except Exception as exc:
        result["exclusion_reason"] = f"Wilcoxon failed: {exc}"
    return result


def condition_series(df: pd.DataFrame, layer: str, stride: int, family: str, metric_col: str) -> pd.Series:
    subset = df[
        (df[COL_LAYER].astype(str) == layer)
        & (pd.to_numeric(df[COL_STRIDE], errors="coerce") == stride)
        & (df[COL_FAMILY].astype(str) == family)
    ]
    if subset.duplicated(subset=[COL_SAMPLE]).any():
        warnings.warn(f"Duplicate sample rows for family={family}, layer={layer}, stride={stride}; keeping first for paired analysis.")
    return subset.drop_duplicates(subset=[COL_SAMPLE]).set_index(COL_SAMPLE)[metric_col]


def format_mean_sd(stats_row: pd.Series) -> str:
    return f"{stats_row['mean']:.4f} ± {stats_row['std']:.4f}"


def format_median_iqr(stats_row: pd.Series) -> str:
    return f"{stats_row['median']:.4f} [{stats_row['q1']:.4f}, {stats_row['q3']:.4f}]"

In [6]:
# Descriptive statistics for every model/layer/stride/metric.
desc_rows = []
group_cols = [COL_FAMILY, COL_MODEL, COL_STRIDE, COL_LAYER]
for keys, group in per_sample.groupby(group_cols, dropna=False):
    group_info = dict(zip(["model_family", "model_id", "temporal_stride", "target_layer"], keys))
    for metric_name, metric_col in metric_columns.items():
        stats_row = descriptive_stats(clean_numeric(group[metric_col]))
        desc_rows.append({
            **group_info,
            "metric": metric_name,
            "metric_column": metric_col,
            "metric_direction": metric_direction[metric_name],
            **stats_row,
        })

descriptive_df = pd.DataFrame(desc_rows)
descriptive_df.to_csv(OUTPUT_TABLE_DIR / "descriptive_statistics.csv", index=False)
descriptive_df.head()

,model_family,model_id,temporal_stride,target_layer,metric,metric_column,metric_direction,n,mean,std,median,q1,q3,iqr,ci95_low,ci95_high
0,convlstm_unet,convlstm_stride_1,1,convlstm_bottleneck_encoder,saliency_consistency,saliency_consistency_mean,higher_better,2552,0.179008,0.157450,0.162613,0.065776,0.279065,0.213288,0.173161,0.185034
1,convlstm_unet,convlstm_stride_1,1,convlstm_bottleneck_encoder,saliency_mask_overlap,center_saliency_mask_overlap,higher_better,2552,0.112364,0.167502,0.053892,0.025463,0.116357,0.090894,0.105762,0.118967
2,convlstm_unet,convlstm_stride_1,1,convlstm_bottleneck_encoder,temporal_saliency_iou,temporal_saliency_iou_mean,higher_better,2552,0.278326,0.147449,0.225984,0.178248,0.362372,0.184124,0.272686,0.284080
3,convlstm_unet,convlstm_stride_1,1,convlstm_bottleneck_encoder,saliency_centroid_motion,saliency_centroid_motion_mean,lower_better,2552,19.941655,7.300558,19.208155,14.699389,24.346232,9.646843,19.660912,20.227206
4,convlstm_unet,convlstm_stride_1,1,convlstm_decoder3_sliding_window,saliency_consistency,saliency_consistency_mean,higher_better,2552,0.996848,0.003432,0.997615,0.996348,0.998431,0.002083,0.996701,0.996975


In [7]:
# A. Final prediction explanations: 2D U-Net final convolution vs ConvLSTM Decoder3 by stride.
FINAL_A_LAYER = "unet_final_convolution"
FINAL_B_LAYER = "convlstm_decoder3_sliding_window"
final_rows = []
for stride in strides:
    for metric_name, metric_col in metric_columns.items():
        a = condition_series(per_sample, FINAL_A_LAYER, stride, "unet_2d", metric_col)
        b = condition_series(per_sample, FINAL_B_LAYER, stride, "convlstm_unet", metric_col)
        matched = sorted(set(a.index) & set(b.index))
        result = paired_wilcoxon_from_vectors(a.loc[matched], b.loc[matched])
        final_rows.append({
            "comparison_family": "final_explanations_unet_vs_decoder3",
            "comparison": "2D U-Net final convolution vs ConvLSTM Decoder3",
            "group_a": "2D U-Net final convolution",
            "group_b": "ConvLSTM Decoder3",
            "direction_convention": "paired difference = group_b - group_a; positive effect means group_b higher",
            "temporal_stride": stride,
            "metric": metric_name,
            "metric_column": metric_col,
            "metric_direction": metric_direction[metric_name],
            "group_a_total_n": int(a.shape[0]),
            "group_b_total_n": int(b.shape[0]),
            "unmatched_group_a_n": int(len(set(a.index) - set(b.index))),
            "unmatched_group_b_n": int(len(set(b.index) - set(a.index))),
            **result,
        })
final_df = pd.DataFrame(final_rows)
final_df["p_holm"], final_df["significant_holm_0_05"] = holm_bonferroni(final_df["p_raw"].tolist())
final_df.to_csv(OUTPUT_TABLE_DIR / "wilcoxon_final_explanations.csv", index=False)
final_df

,comparison_family,comparison,group_a,group_b,direction_convention,temporal_stride,metric,metric_column,metric_direction,group_a_total_n,...,paired_n,zero_differences,median_paired_difference_b_minus_a,rank_biserial_b_minus_a,wilcoxon_statistic,p_raw,test_performed,exclusion_reason,p_holm,significant_holm_0_05
0,final_explanations_unet_vs_decoder3,2D U-Net final convolution vs ConvLSTM Decoder3,2D U-Net final convolution,ConvLSTM Decoder3,paired difference = group_b - group_a; positiv...,1,saliency_consistency,saliency_consistency_mean,higher_better,2552,...,2552,0,0.003240,0.987952,19624.0,0.000000e+00,True,,0.000000e+00,True
1,final_explanations_unet_vs_decoder3,2D U-Net final convolution vs ConvLSTM Decoder3,2D U-Net final convolution,ConvLSTM Decoder3,paired difference = group_b - group_a; positiv...,1,saliency_mask_overlap,center_saliency_mask_overlap,higher_better,2552,...,2552,0,-0.060113,-0.987926,19667.0,0.000000e+00,True,,0.000000e+00,True
2,final_explanations_unet_vs_decoder3,2D U-Net final convolution vs ConvLSTM Decoder3,2D U-Net final convolution,ConvLSTM Decoder3,paired difference = group_b - group_a; positiv...,1,temporal_saliency_iou,temporal_saliency_iou_mean,higher_better,2552,...,2552,290,-0.055195,-0.955112,57445.0,0.000000e+00,True,,0.000000e+00,True
3,final_explanations_unet_vs_decoder3,2D U-Net final convolution vs ConvLSTM Decoder3,2D U-Net final convolution,ConvLSTM Decoder3,paired difference = group_b - group_a; positiv...,1,saliency_centroid_motion,saliency_centroid_motion_mean,lower_better,2552,...,2552,0,-0.093864,-0.547557,736945.0,7.702216e-127,True,,1.540443e-126,True
4,final_explanations_unet_vs_decoder3,2D U-Net final convolution vs ConvLSTM Decoder3,2D U-Net final convolution,ConvLSTM Decoder3,paired difference = group_b - group_a; positiv...,4,saliency_consistency,saliency_consistency_mean,higher_better,2552,...,2552,0,0.017150,0.984472,25292.0,0.000000e+00,True,,0.000000e+00,True
5,final_explanations_unet_vs_decoder3,2D U-Net final convolution vs ConvLSTM Decoder3,2D U-Net final convolution,ConvLSTM Decoder3,paired difference = group_b - group_a; positiv...,4,saliency_mask_overlap,center_saliency_mask_overlap,higher_better,2552,...,2552,0,-0.051157,-0.983435,26982.0,0.000000e+00,True,,0.000000e+00,True
6,final_explanations_unet_vs_decoder3,2D U-Net final convolution vs ConvLSTM Decoder3,2D U-Net final convolution,ConvLSTM Decoder3,paired difference = group_b - group_a; positiv...,4,temporal_saliency_iou,temporal_saliency_iou_mean,higher_better,2552,...,2552,1483,0.000000,-0.888494,31886.0,9.735062e-140,True,,2.920519e-139,True
7,final_explanations_unet_vs_decoder3,2D U-Net final convolution vs ConvLSTM Decoder3,2D U-Net final convolution,ConvLSTM Decoder3,paired difference = group_b - group_a; positiv...,4,saliency_centroid_motion,saliency_centroid_motion_mean,lower_better,2552,...,2552,0,-0.367829,-0.860850,226650.0,1.841788e-310,True,,1.841788e-309,True
8,final_explanations_unet_vs_decoder3,2D U-Net final convolution vs ConvLSTM Decoder3,2D U-Net final convolution,ConvLSTM Decoder3,paired difference = group_b - group_a; positiv...,6,saliency_consistency,saliency_consistency_mean,higher_better,2552,...,2552,0,0.012077,0.752230,403572.0,1.445222e-237,True,,1.011655e-236,True
9,final_explanations_unet_vs_decoder3,2D U-Net final convolution vs ConvLSTM Decoder3,2D U-Net final convolution,ConvLSTM Decoder3,paired difference = group_b - group_a; positiv...,6,saliency_mask_overlap,center_saliency_mask_overlap,higher_better,2552,...,2552,0,-0.273776,-0.999892,176.0,0.000000e+00,True,,0.000000e+00,True


In [8]:
# B. Internal temporal representations: ConvLSTM encoder bottleneck vs temporal bottleneck by stride.
INTERNAL_A_LAYER = "convlstm_bottleneck_encoder"
INTERNAL_B_LAYER = "convlstm_temporal_bottleneck"
internal_rows = []
for stride in strides:
    for metric_name, metric_col in metric_columns.items():
        a = condition_series(per_sample, INTERNAL_A_LAYER, stride, "convlstm_unet", metric_col)
        b = condition_series(per_sample, INTERNAL_B_LAYER, stride, "convlstm_unet", metric_col)
        matched = sorted(set(a.index) & set(b.index))
        result = paired_wilcoxon_from_vectors(a.loc[matched], b.loc[matched])
        internal_rows.append({
            "comparison_family": "internal_encoder_vs_temporal_bottleneck",
            "comparison": "ConvLSTM encoder bottleneck vs temporal bottleneck",
            "group_a": "ConvLSTM encoder bottleneck",
            "group_b": "ConvLSTM temporal bottleneck",
            "direction_convention": "paired difference = group_b - group_a; positive effect means temporal bottleneck higher",
            "temporal_stride": stride,
            "metric": metric_name,
            "metric_column": metric_col,
            "metric_direction": metric_direction[metric_name],
            "group_a_total_n": int(a.shape[0]),
            "group_b_total_n": int(b.shape[0]),
            "unmatched_group_a_n": int(len(set(a.index) - set(b.index))),
            "unmatched_group_b_n": int(len(set(b.index) - set(a.index))),
            **result,
        })
internal_df = pd.DataFrame(internal_rows)
internal_df["p_holm"], internal_df["significant_holm_0_05"] = holm_bonferroni(internal_df["p_raw"].tolist())
internal_df.to_csv(OUTPUT_TABLE_DIR / "wilcoxon_internal_representations.csv", index=False)
internal_df

,comparison_family,comparison,group_a,group_b,direction_convention,temporal_stride,metric,metric_column,metric_direction,group_a_total_n,...,paired_n,zero_differences,median_paired_difference_b_minus_a,rank_biserial_b_minus_a,wilcoxon_statistic,p_raw,test_performed,exclusion_reason,p_holm,significant_holm_0_05
0,internal_encoder_vs_temporal_bottleneck,ConvLSTM encoder bottleneck vs temporal bottle...,ConvLSTM encoder bottleneck,ConvLSTM temporal bottleneck,paired difference = group_b - group_a; positiv...,1,saliency_consistency,saliency_consistency_mean,higher_better,2552,...,2552,0,0.129508,0.730268,439343.0,5.093907e-224,True,,7.131469e-223,True
1,internal_encoder_vs_temporal_bottleneck,ConvLSTM encoder bottleneck vs temporal bottle...,ConvLSTM encoder bottleneck,ConvLSTM temporal bottleneck,paired difference = group_b - group_a; positiv...,1,saliency_mask_overlap,center_saliency_mask_overlap,higher_better,2552,...,2552,0,-0.025147,-0.558855,718543.0,4.802906e-132,True,,4.802906e-131,True
2,internal_encoder_vs_temporal_bottleneck,ConvLSTM encoder bottleneck vs temporal bottle...,ConvLSTM encoder bottleneck,ConvLSTM temporal bottleneck,paired difference = group_b - group_a; positiv...,1,temporal_saliency_iou,temporal_saliency_iou_mean,higher_better,2552,...,2552,0,0.035500,0.230100,1254024.0,7.675673e-24,True,,3.070269e-23,True
3,internal_encoder_vs_temporal_bottleneck,ConvLSTM encoder bottleneck vs temporal bottle...,ConvLSTM encoder bottleneck,ConvLSTM temporal bottleneck,paired difference = group_b - group_a; positiv...,1,saliency_centroid_motion,saliency_centroid_motion_mean,lower_better,2552,...,2552,0,-8.269097,-0.914406,139417.0,0.000000e+00,True,,0.000000e+00,True
4,internal_encoder_vs_temporal_bottleneck,ConvLSTM encoder bottleneck vs temporal bottle...,ConvLSTM encoder bottleneck,ConvLSTM temporal bottleneck,paired difference = group_b - group_a; positiv...,4,saliency_consistency,saliency_consistency_mean,higher_better,2552,...,2552,0,0.100848,0.658225,556688.0,2.164129e-182,True,,2.813368e-181,True
5,internal_encoder_vs_temporal_bottleneck,ConvLSTM encoder bottleneck vs temporal bottle...,ConvLSTM encoder bottleneck,ConvLSTM temporal bottleneck,paired difference = group_b - group_a; positiv...,4,saliency_mask_overlap,center_saliency_mask_overlap,higher_better,2552,...,2552,0,-0.078395,-0.841665,257899.0,7.129212e-297,True,,1.140674e-295,True
6,internal_encoder_vs_temporal_bottleneck,ConvLSTM encoder bottleneck vs temporal bottle...,ConvLSTM encoder bottleneck,ConvLSTM temporal bottleneck,paired difference = group_b - group_a; positiv...,4,temporal_saliency_iou,temporal_saliency_iou_mean,higher_better,2552,...,2552,0,-0.000255,-0.081543,1495995.5,3.599903e-04,True,,3.599903e-04,True
7,internal_encoder_vs_temporal_bottleneck,ConvLSTM encoder bottleneck vs temporal bottle...,ConvLSTM encoder bottleneck,ConvLSTM temporal bottleneck,paired difference = group_b - group_a; positiv...,4,saliency_centroid_motion,saliency_centroid_motion_mean,lower_better,2552,...,2552,0,-8.256609,-0.911122,144766.0,0.000000e+00,True,,0.000000e+00,True
8,internal_encoder_vs_temporal_bottleneck,ConvLSTM encoder bottleneck vs temporal bottle...,ConvLSTM encoder bottleneck,ConvLSTM temporal bottleneck,paired difference = group_b - group_a; positiv...,6,saliency_consistency,saliency_consistency_mean,higher_better,2552,...,2552,0,0.068784,0.378896,1011663.0,1.004354e-61,True,,6.026125e-61,True
9,internal_encoder_vs_temporal_bottleneck,ConvLSTM encoder bottleneck vs temporal bottle...,ConvLSTM encoder bottleneck,ConvLSTM temporal bottleneck,paired difference = group_b - group_a; positiv...,6,saliency_mask_overlap,center_saliency_mask_overlap,higher_better,2552,...,2552,0,0.028841,0.237386,1242157.0,2.857396e-25,True,,1.428698e-24,True


In [9]:
# C. Effect of temporal stride: Friedman repeated-measures tests and significant post-hoc Wilcoxon tests.
stride_layers = [
    "convlstm_decoder3_sliding_window",
    "convlstm_bottleneck_encoder",
    "convlstm_temporal_bottleneck",
]
friedman_rows = []
posthoc_rows = []

for layer in stride_layers:
    for metric_name, metric_col in metric_columns.items():
        wide = []
        valid_strides = []
        for stride in strides:
            s = condition_series(per_sample, layer, stride, "convlstm_unet", metric_col).rename(stride)
            wide.append(s)
            valid_strides.append(stride)
        pivot = pd.concat(wide, axis=1, join="inner")
        pivot.columns = valid_strides
        pivot = pivot.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna(axis=0, how="any")
        row = {
            "comparison_family": "stride_effect_friedman",
            "target_layer": layer,
            "metric": metric_name,
            "metric_column": metric_col,
            "metric_direction": metric_direction[metric_name],
            "strides": json.dumps(valid_strides),
            "paired_n_complete_cases": int(len(pivot)),
            "friedman_statistic": np.nan,
            "p_raw": np.nan,
            "test_performed": False,
            "exclusion_reason": "",
        }
        if len(pivot) < 5:
            row["exclusion_reason"] = f"Too few complete repeated-measures cases: {len(pivot)} < 5"
        elif len(valid_strides) < 3:
            row["exclusion_reason"] = f"Friedman requires at least 3 related groups; found {len(valid_strides)}"
        else:
            try:
                fried = stats.friedmanchisquare(*[pivot[stride].to_numpy(dtype=float) for stride in valid_strides])
                row["friedman_statistic"] = float(fried.statistic)
                row["p_raw"] = float(fried.pvalue)
                row["test_performed"] = True
            except Exception as exc:
                row["exclusion_reason"] = f"Friedman failed: {exc}"
        friedman_rows.append(row)

        if row["test_performed"] and row["p_raw"] < ALPHA:
            family_start = len(posthoc_rows)
            for stride_a, stride_b in itertools.combinations(valid_strides, 2):
                result = paired_wilcoxon_from_vectors(pivot[stride_a], pivot[stride_b])
                posthoc_rows.append({
                    "comparison_family": f"stride_posthoc_{layer}_{metric_name}",
                    "comparison": f"Stride {stride_a} vs stride {stride_b}",
                    "target_layer": layer,
                    "group_a": f"stride_{stride_a}",
                    "group_b": f"stride_{stride_b}",
                    "direction_convention": "paired difference = group_b - group_a; positive effect means higher value at group_b stride",
                    "stride_a": stride_a,
                    "stride_b": stride_b,
                    "metric": metric_name,
                    "metric_column": metric_col,
                    "metric_direction": metric_direction[metric_name],
                    **result,
                })
            family_indices = list(range(family_start, len(posthoc_rows)))
            adjusted, reject = holm_bonferroni([posthoc_rows[i]["p_raw"] for i in family_indices])
            for local_idx, row_idx in enumerate(family_indices):
                posthoc_rows[row_idx]["p_holm"] = adjusted[local_idx]
                posthoc_rows[row_idx]["significant_holm_0_05"] = reject[local_idx]

friedman_df = pd.DataFrame(friedman_rows)
posthoc_stride_df = pd.DataFrame(posthoc_rows)
friedman_df.to_csv(OUTPUT_TABLE_DIR / "friedman_stride_effects.csv", index=False)
posthoc_stride_df.to_csv(OUTPUT_TABLE_DIR / "wilcoxon_stride_posthoc.csv", index=False)
display(friedman_df)
display(posthoc_stride_df.head())

,comparison_family,target_layer,metric,metric_column,metric_direction,strides,paired_n_complete_cases,friedman_statistic,p_raw,test_performed,exclusion_reason
0,stride_effect_friedman,convlstm_decoder3_sliding_window,saliency_consistency,saliency_consistency_mean,higher_better,"[1, 4, 6, 8, 10]",2552,8964.455799,0.000000e+00,True,
1,stride_effect_friedman,convlstm_decoder3_sliding_window,saliency_mask_overlap,center_saliency_mask_overlap,higher_better,"[1, 4, 6, 8, 10]",2552,6264.174608,0.000000e+00,True,
2,stride_effect_friedman,convlstm_decoder3_sliding_window,temporal_saliency_iou,temporal_saliency_iou_mean,higher_better,"[1, 4, 6, 8, 10]",2552,1890.600508,0.000000e+00,True,
3,stride_effect_friedman,convlstm_decoder3_sliding_window,saliency_centroid_motion,saliency_centroid_motion_mean,lower_better,"[1, 4, 6, 8, 10]",2552,6985.967712,0.000000e+00,True,
4,stride_effect_friedman,convlstm_bottleneck_encoder,saliency_consistency,saliency_consistency_mean,higher_better,"[1, 4, 6, 8, 10]",2551,445.206742,4.721311e-95,True,
5,stride_effect_friedman,convlstm_bottleneck_encoder,saliency_mask_overlap,center_saliency_mask_overlap,higher_better,"[1, 4, 6, 8, 10]",2552,1325.281657,1.098191e-285,True,
6,stride_effect_friedman,convlstm_bottleneck_encoder,temporal_saliency_iou,temporal_saliency_iou_mean,higher_better,"[1, 4, 6, 8, 10]",2552,695.249912,3.721846e-149,True,
7,stride_effect_friedman,convlstm_bottleneck_encoder,saliency_centroid_motion,saliency_centroid_motion_mean,lower_better,"[1, 4, 6, 8, 10]",2551,311.947628,2.865896e-66,True,
8,stride_effect_friedman,convlstm_temporal_bottleneck,saliency_consistency,saliency_consistency_mean,higher_better,"[1, 4, 6, 8, 10]",2552,1167.532288,1.740071e-251,True,
9,stride_effect_friedman,convlstm_temporal_bottleneck,saliency_mask_overlap,center_saliency_mask_overlap,higher_better,"[1, 4, 6, 8, 10]",2552,3140.414420,0.000000e+00,True,


,comparison_family,comparison,target_layer,group_a,group_b,direction_convention,stride_a,stride_b,metric,metric_column,...,paired_n,zero_differences,median_paired_difference_b_minus_a,rank_biserial_b_minus_a,wilcoxon_statistic,p_raw,test_performed,exclusion_reason,p_holm,significant_holm_0_05
0,stride_posthoc_convlstm_decoder3_sliding_windo...,Stride 1 vs stride 4,convlstm_decoder3_sliding_window,stride_1,stride_4,paired difference = group_b - group_a; positiv...,1,4,saliency_consistency,saliency_consistency_mean,...,2552,0,-0.021660,-0.998351,2686.0,0.0,True,,0.0,True
1,stride_posthoc_convlstm_decoder3_sliding_windo...,Stride 1 vs stride 6,convlstm_decoder3_sliding_window,stride_1,stride_6,paired difference = group_b - group_a; positiv...,1,6,saliency_consistency,saliency_consistency_mean,...,2552,0,-0.052730,-0.999183,1331.0,0.0,True,,0.0,True
2,stride_posthoc_convlstm_decoder3_sliding_windo...,Stride 1 vs stride 8,convlstm_decoder3_sliding_window,stride_1,stride_8,paired difference = group_b - group_a; positiv...,1,8,saliency_consistency,saliency_consistency_mean,...,2552,0,-0.064639,-0.999995,8.0,0.0,True,,0.0,True
3,stride_posthoc_convlstm_decoder3_sliding_windo...,Stride 1 vs stride 10,convlstm_decoder3_sliding_window,stride_1,stride_10,paired difference = group_b - group_a; positiv...,1,10,saliency_consistency,saliency_consistency_mean,...,2552,0,-0.075133,-0.999980,33.0,0.0,True,,0.0,True
4,stride_posthoc_convlstm_decoder3_sliding_windo...,Stride 4 vs stride 6,convlstm_decoder3_sliding_window,stride_4,stride_6,paired difference = group_b - group_a; positiv...,4,6,saliency_consistency,saliency_consistency_mean,...,2552,0,-0.030124,-0.999296,1147.0,0.0,True,,0.0,True


In [10]:
# Publication summary table combining main paired Wilcoxon comparisons and significant stride post-hoc tests.
def stats_lookup(group_label: str, layer: str, stride: int, family: str, metric_name: str) -> pd.Series:
    metric_col = metric_columns[metric_name]
    subset = per_sample[
        (per_sample[COL_LAYER].astype(str) == layer)
        & (pd.to_numeric(per_sample[COL_STRIDE], errors="coerce") == stride)
        & (per_sample[COL_FAMILY].astype(str) == family)
    ]
    stats_dict = descriptive_stats(clean_numeric(subset[metric_col]))
    return pd.Series(stats_dict)

publication_rows = []
for source_name, df in [("final_explanations", final_df), ("internal_representations", internal_df)]:
    for row in df.itertuples(index=False):
        metric_name = row.metric
        stride = int(row.temporal_stride)
        if source_name == "final_explanations":
            a_stats = stats_lookup(row.group_a, FINAL_A_LAYER, stride, "unet_2d", metric_name)
            b_stats = stats_lookup(row.group_b, FINAL_B_LAYER, stride, "convlstm_unet", metric_name)
        else:
            a_stats = stats_lookup(row.group_a, INTERNAL_A_LAYER, stride, "convlstm_unet", metric_name)
            b_stats = stats_lookup(row.group_b, INTERNAL_B_LAYER, stride, "convlstm_unet", metric_name)
        publication_rows.append({
            "comparison_family": row.comparison_family,
            "comparison": row.comparison,
            "temporal_stride": stride,
            "metric": metric_name,
            "metric_direction": row.metric_direction,
            "group_a": row.group_a,
            "group_a_mean_sd": format_mean_sd(a_stats),
            "group_a_median_iqr": format_median_iqr(a_stats),
            "group_b": row.group_b,
            "group_b_mean_sd": format_mean_sd(b_stats),
            "group_b_median_iqr": format_median_iqr(b_stats),
            "paired_n": row.paired_n,
            "median_paired_difference_b_minus_a": row.median_paired_difference_b_minus_a,
            "wilcoxon_statistic": row.wilcoxon_statistic,
            "p_raw": row.p_raw,
            "p_holm": row.p_holm,
            "matched_pairs_rank_biserial_b_minus_a": row.rank_biserial_b_minus_a,
            "significant_holm_0_05": row.significant_holm_0_05,
            "direction_convention": row.direction_convention,
            "test_performed": row.test_performed,
            "exclusion_reason": row.exclusion_reason,
        })

if not posthoc_stride_df.empty:
    for row in posthoc_stride_df.itertuples(index=False):
        publication_rows.append({
            "comparison_family": row.comparison_family,
            "comparison": f"{row.target_layer}: {row.comparison}",
            "temporal_stride": f"{row.stride_a}_vs_{row.stride_b}",
            "metric": row.metric,
            "metric_direction": row.metric_direction,
            "group_a": row.group_a,
            "group_a_mean_sd": "see descriptive_statistics.csv",
            "group_a_median_iqr": "see descriptive_statistics.csv",
            "group_b": row.group_b,
            "group_b_mean_sd": "see descriptive_statistics.csv",
            "group_b_median_iqr": "see descriptive_statistics.csv",
            "paired_n": row.paired_n,
            "median_paired_difference_b_minus_a": row.median_paired_difference_b_minus_a,
            "wilcoxon_statistic": row.wilcoxon_statistic,
            "p_raw": row.p_raw,
            "p_holm": row.p_holm,
            "matched_pairs_rank_biserial_b_minus_a": row.rank_biserial_b_minus_a,
            "significant_holm_0_05": row.significant_holm_0_05,
            "direction_convention": row.direction_convention,
            "test_performed": row.test_performed,
            "exclusion_reason": row.exclusion_reason,
        })

publication_df = pd.DataFrame(publication_rows)
publication_df.to_csv(OUTPUT_TABLE_DIR / "publication_summary_table.csv", index=False)
latex_cols = [
    "comparison", "temporal_stride", "metric", "group_a_mean_sd", "group_a_median_iqr",
    "group_b_mean_sd", "group_b_median_iqr", "paired_n", "median_paired_difference_b_minus_a",
    "wilcoxon_statistic", "p_raw", "p_holm", "matched_pairs_rank_biserial_b_minus_a", "significant_holm_0_05",
]
with (OUTPUT_TABLE_DIR / "publication_summary_table.tex").open("w", encoding="utf-8") as f:
    f.write(publication_df[latex_cols].to_latex(index=False, float_format="%.4g", escape=True))
publication_df.head()

,comparison_family,comparison,temporal_stride,metric,metric_direction,group_a,group_a_mean_sd,group_a_median_iqr,group_b,group_b_mean_sd,...,paired_n,median_paired_difference_b_minus_a,wilcoxon_statistic,p_raw,p_holm,matched_pairs_rank_biserial_b_minus_a,significant_holm_0_05,direction_convention,test_performed,exclusion_reason
0,final_explanations_unet_vs_decoder3,2D U-Net final convolution vs ConvLSTM Decoder3,1,saliency_consistency,higher_better,2D U-Net final convolution,0.9912 ± 0.0124,"0.9941 [0.9906, 0.9963]",ConvLSTM Decoder3,0.9968 ± 0.0034,...,2552,0.003240,19624.0,0.000000e+00,0.000000e+00,0.987952,True,paired difference = group_b - group_a; positiv...,True,
1,final_explanations_unet_vs_decoder3,2D U-Net final convolution vs ConvLSTM Decoder3,1,saliency_mask_overlap,higher_better,2D U-Net final convolution,0.9612 ± 0.0448,"0.9726 [0.9493, 0.9875]",ConvLSTM Decoder3,0.8963 ± 0.0526,...,2552,-0.060113,19667.0,0.000000e+00,0.000000e+00,-0.987926,True,paired difference = group_b - group_a; positiv...,True,
2,final_explanations_unet_vs_decoder3,2D U-Net final convolution vs ConvLSTM Decoder3,1,temporal_saliency_iou,higher_better,2D U-Net final convolution,0.9960 ± 0.0341,"1.0000 [1.0000, 1.0000]",ConvLSTM Decoder3,0.9061 ± 0.1199,...,2552,-0.055195,57445.0,0.000000e+00,0.000000e+00,-0.955112,True,paired difference = group_b - group_a; positiv...,True,
3,final_explanations_unet_vs_decoder3,2D U-Net final convolution vs ConvLSTM Decoder3,1,saliency_centroid_motion,lower_better,2D U-Net final convolution,0.6077 ± 0.5603,"0.4781 [0.3443, 0.6889]",ConvLSTM Decoder3,0.4375 ± 0.2624,...,2552,-0.093864,736945.0,7.702216e-127,1.540443e-126,-0.547557,True,paired difference = group_b - group_a; positiv...,True,
4,final_explanations_unet_vs_decoder3,2D U-Net final convolution vs ConvLSTM Decoder3,4,saliency_consistency,higher_better,2D U-Net final convolution,0.9495 ± 0.0331,"0.9571 [0.9383, 0.9708]",ConvLSTM Decoder3,0.9733 ± 0.0139,...,2552,0.017150,25292.0,0.000000e+00,0.000000e+00,0.984472,True,paired difference = group_b - group_a; positiv...,True,


In [11]:
# Publication-ready boxplots.
DISPLAY_LAYER_NAMES = {
    "unet_final_convolution": "2D U-Net final conv",
    "unet_encoder_bottleneck": "2D U-Net encoder bottleneck",
    "convlstm_decoder3_sliding_window": "ConvLSTM Decoder3",
    "convlstm_bottleneck_encoder": "ConvLSTM encoder bottleneck",
    "convlstm_temporal_bottleneck": "ConvLSTM temporal bottleneck",
}
DISPLAY_METRIC_NAMES = {
    "saliency_consistency": "Saliency consistency",
    "saliency_mask_overlap": "Center saliency-mask overlap",
    "temporal_saliency_iou": "Temporal saliency IoU",
    "saliency_centroid_motion": "Saliency centroid motion (pixels)",
}


def add_boxplot(ax, data_groups: list[np.ndarray], labels: list[str], title: str, ylabel: str, jitter: bool = True) -> None:
    clean_groups = [np.asarray(g, dtype=float)[np.isfinite(np.asarray(g, dtype=float))] for g in data_groups]
    bp = ax.boxplot(clean_groups, tick_labels=labels, showfliers=False, patch_artist=True, medianprops={"color": "black"})
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(clean_groups), 3)))
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.65)
    if jitter:
        for i, values in enumerate(clean_groups, start=1):
            if len(values) <= 400:
                x = np.random.default_rng(RANDOM_SEED + i).normal(i, 0.04, size=len(values))
                ax.scatter(x, values, s=5, alpha=0.18, color="black", linewidths=0)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.25)


def save_figure(fig: plt.Figure, stem: str) -> None:
    png_path = OUTPUT_FIGURE_DIR / f"{stem}.png"
    pdf_path = OUTPUT_FIGURE_DIR / f"{stem}.pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)

for metric_name, metric_col in metric_columns.items():
    # Final explanation figures: rows/axes by stride, 2D U-Net final vs ConvLSTM Decoder3.
    fig, axes = plt.subplots(1, len(strides), figsize=(4 * len(strides), 4.8), sharey=False)
    if len(strides) == 1:
        axes = [axes]
    for ax, stride in zip(axes, strides):
        a = condition_series(per_sample, FINAL_A_LAYER, stride, "unet_2d", metric_col)
        b = condition_series(per_sample, FINAL_B_LAYER, stride, "convlstm_unet", metric_col)
        matched = sorted(set(a.index) & set(b.index))
        add_boxplot(
            ax,
            [a.loc[matched].to_numpy(dtype=float), b.loc[matched].to_numpy(dtype=float)],
            ["2D U-Net", "ConvLSTM\nDecoder3"],
            f"Stride {stride}\nn={len(matched)}",
            DISPLAY_METRIC_NAMES[metric_name],
        )
    fig.suptitle(f"Final prediction explanations: {DISPLAY_METRIC_NAMES[metric_name]}", y=1.04)
    save_figure(fig, f"boxplot_final_explanations_{metric_name}")

    # Internal representation figures: encoder bottleneck vs temporal bottleneck by stride.
    fig, axes = plt.subplots(1, len(strides), figsize=(4 * len(strides), 4.8), sharey=False)
    if len(strides) == 1:
        axes = [axes]
    for ax, stride in zip(axes, strides):
        a = condition_series(per_sample, INTERNAL_A_LAYER, stride, "convlstm_unet", metric_col)
        b = condition_series(per_sample, INTERNAL_B_LAYER, stride, "convlstm_unet", metric_col)
        matched = sorted(set(a.index) & set(b.index))
        add_boxplot(
            ax,
            [a.loc[matched].to_numpy(dtype=float), b.loc[matched].to_numpy(dtype=float)],
            ["Encoder\nbottleneck", "Temporal\nbottleneck"],
            f"Stride {stride}\nn={len(matched)}",
            DISPLAY_METRIC_NAMES[metric_name],
        )
    fig.suptitle(f"Internal temporal representations: {DISPLAY_METRIC_NAMES[metric_name]}", y=1.04)
    save_figure(fig, f"boxplot_internal_representations_{metric_name}")

print("Saved boxplots to", OUTPUT_FIGURE_DIR)

Saved boxplots to outputs/iMIMIC_additional_experiments/iMIMIC_updated_gradcam_temporal_evaluation_statistical_analysis_07_28/figures


In [12]:
# Copy required top-level outputs to the requested output root as well as tables/.
for name in [
    "descriptive_statistics.csv",
    "wilcoxon_final_explanations.csv",
    "wilcoxon_internal_representations.csv",
    "publication_summary_table.csv",
    "publication_summary_table.tex",
]:
    src = OUTPUT_TABLE_DIR / name
    dst = OUTPUT_DIR / name
    if src.exists():
        dst.write_bytes(src.read_bytes())

src_report = OUTPUT_REPORT_DIR / "data_validation_report.txt"
if src_report.exists():
    (OUTPUT_DIR / "data_validation_report.txt").write_text(src_report.read_text(encoding="utf-8"), encoding="utf-8")

summary = {
    "input_run_dir": str(INPUT_RUN_DIR),
    "per_sample_path": str(PER_SAMPLE_PATH),
    "output_dir": str(OUTPUT_DIR),
    "metric_columns": metric_columns,
    "metric_direction": metric_direction,
    "alpha": ALPHA,
    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
    "random_seed": RANDOM_SEED,
    "wilcoxon_zero_method": "wilcox",
    "effect_size": "matched-pairs rank-biserial correlation, paired difference group_b - group_a",
}
(OUTPUT_DIR / "analysis_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("Done. Outputs written to:", OUTPUT_DIR)

{
  "input_run_dir": "outputs/iMIMIC_additional_experiments/iMIMIC_updated_gradcam_temporal_evaluation_07_26",
  "per_sample_path": "outputs/iMIMIC_additional_experiments/iMIMIC_updated_gradcam_temporal_evaluation_07_26/metrics/per_sample_metrics.csv",
  "output_dir": "outputs/iMIMIC_additional_experiments/iMIMIC_updated_gradcam_temporal_evaluation_statistical_analysis_07_28",
  "metric_columns": {
    "saliency_consistency": "saliency_consistency_mean",
    "saliency_mask_overlap": "center_saliency_mask_overlap",
    "temporal_saliency_iou": "temporal_saliency_iou_mean",
    "saliency_centroid_motion": "saliency_centroid_motion_mean"
  },
  "metric_direction": {
    "saliency_consistency": "higher_better",
    "saliency_mask_overlap": "higher_better",
    "temporal_saliency_iou": "higher_better",
    "saliency_centroid_motion": "lower_better"
  },
  "alpha": 0.05,
  "bootstrap_iterations": 5000,
  "random_seed": 42,
  "wilcoxon_zero_method": "wilcox",
  "effect_size": "matched-pairs r